# CELL 1 — Install packages

In [1]:
!pip install transformers datasets vaderSentiment textblob wordcloud accelerate -q
!python -m textblob.download_corpora
print("✅ Packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.7 MB/s eta 0:00:00
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
Finished.
✅ Packages installed


# CELL 2 — Mount Drive + Imports

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import os, time, warnings
warnings.filterwarnings('ignore')

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from scipy.special import softmax

# ── Paths
DRIVE_PATH  = '/content/drive/MyDrive/PoliGraphX'
OUTPUT_PATH = os.path.join(DRIVE_PATH, 'outputs')
os.makedirs(OUTPUT_PATH, exist_ok=True)

print("✅ Imports done")
print(f"CUDA: {torch.cuda.is_available()}")

Mounted at /content/drive
✅ Imports done
CUDA: True


# CELL 3 — Load labeled tweets

In [3]:
print("Loading tweets_labeled.csv...")

df = pd.read_csv(os.path.join(DRIVE_PATH, 'tweets_labeled.csv'), low_memory=False)
df = df[df['clean_text'].notna()].copy()
df = df[df['clean_text'].str.strip() != ''].copy()
df = df.reset_index(drop=True)

print(f"Total tweets : {len(df):,}")
print(f"Columns      : {list(df.columns)}")
print(f"\nIdeology distribution:")
print(df['ideology'].value_counts())

# Parse date if available
if 'date' in df.columns or 'created_at' in df.columns:
    date_col = 'date' if 'date' in df.columns else 'created_at'
    df['date'] = pd.to_datetime(df[date_col], errors='coerce')
    df['month'] = df['date'].dt.to_period('M')
    print(f"\nDate range: {df['date'].min()} → {df['date'].max()}")
else:
    print("\n⚠️ No date column found — time-based graph will be skipped")

print("✅ Data loaded")

Loading tweets_labeled.csv...
Total tweets : 42,110
Columns      : ['text', 'rawContent', 'date', 'lang', 'username', 'likeCount', 'retweetCount', 'replyCount', 'quoteCount', 'viewCount', 'hashtags', 'mentionedUsers', 'retweetedTweet', 'epoch', 'id_str', 'location', 'tweet_text', 'date_only', 'year', 'month', 'day', 'hour', 'hashtags_list', 'hashtags_str', 'hashtag_count', 'mentioned_list', 'mention_count', 'clean_text', 'ideology', 'ideology_confidence']

Ideology distribution:
ideology
Right    26866
Left     15244
Name: count, dtype: int64

Date range: 2024-11-26 00:00:00 → 2024-11-26 00:00:00
✅ Data loaded


# CELL 4 — Method 1: VADER Sentiment

In [4]:
print("Running VADER Sentiment Analysis...")
t_start = time.time()

analyzer = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    score = analyzer.polarity_scores(str(text))['compound']
    return 'Positive' if score >= 0 else 'Negative'

def vader_confidence(text):
    score = analyzer.polarity_scores(str(text))['compound']
    return round(abs(score), 4)

df['vader_sentiment']   = df['clean_text'].apply(vader_sentiment)
df['vader_confidence']  = df['clean_text'].apply(vader_confidence)

print(f"Done in {(time.time()-t_start)/60:.1f} minutes")
print(f"\nVADER Distribution:")
print(df['vader_sentiment'].value_counts())
print("✅ VADER complete")

Running VADER Sentiment Analysis...
Done in 0.3 minutes

VADER Distribution:
vader_sentiment
Positive    26560
Negative    15550
Name: count, dtype: int64
✅ VADER complete


# CELL 5 — Method 2: TextBlob Sentiment

In [5]:
print("Running TextBlob Sentiment Analysis...")
t_start = time.time()

def textblob_sentiment(text):
    score = TextBlob(str(text)).sentiment.polarity
    return 'Positive' if score >= 0 else 'Negative'

def textblob_confidence(text):
    score = TextBlob(str(text)).sentiment.polarity
    return round(abs(score), 4)

df['textblob_sentiment']  = df['clean_text'].apply(textblob_sentiment)
df['textblob_confidence'] = df['clean_text'].apply(textblob_confidence)

print(f"Done in {(time.time()-t_start)/60:.1f} minutes")
print(f"\nTextBlob Distribution:")
print(df['textblob_sentiment'].value_counts())
print("✅ TextBlob complete")

Running TextBlob Sentiment Analysis...
Done in 0.4 minutes

TextBlob Distribution:
textblob_sentiment
Positive    32004
Negative    10106
Name: count, dtype: int64
✅ TextBlob complete


# CELL 6 — Method 3: RoBERTa Sentiment (Twitter-trained)

In [6]:
print("Loading RoBERTa Sentiment Model...")
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer_s = AutoTokenizer.from_pretrained(MODEL_NAME)
model_s     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_s     = model_s.to(device)
model_s.eval()

# Labels for this model: 0=Negative, 1=Neutral, 2=Positive
# We map Neutral → Negative for Positive/Negative only
ROBERTA_LABELS = {0: 'Negative', 1: 'Negative', 2: 'Positive'}

print("Running RoBERTa inference...")
t_start = time.time()

BATCH_SIZE   = 64
all_sentiments = []
all_confidences = []

for i in range(0, len(df), BATCH_SIZE):
    batch = df['clean_text'].iloc[i:i+BATCH_SIZE].astype(str).str[:512].tolist()
    enc   = tokenizer_s(batch, truncation=True, padding=True,
                        max_length=128, return_tensors='pt').to(device)
    with torch.no_grad():
        logits = model_s(**enc).logits
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()
    preds  = np.argmax(probs, axis=-1)
    confs  = np.max(probs, axis=-1)

    all_sentiments.extend([ROBERTA_LABELS[p] for p in preds])
    all_confidences.extend([round(float(c), 4) for c in confs])

    done = min(i + BATCH_SIZE, len(df))
    print(f"  {done:,}/{len(df):,} ({done/len(df)*100:.1f}%)", end='\r')

df['roberta_sentiment']   = all_sentiments
df['roberta_confidence']  = all_confidences

print(f"\nDone in {(time.time()-t_start)/60:.1f} minutes")
print(f"\nRoBERTa Distribution:")
print(df['roberta_sentiment'].value_counts())
print("✅ RoBERTa Sentiment complete")

Loading RoBERTa Sentiment Model...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running RoBERTa inference...
  42,110/42,110 (100.0%)
Done in 3.4 minutes

RoBERTa Distribution:
roberta_sentiment
Negative    35660
Positive     6450
Name: count, dtype: int64
✅ RoBERTa Sentiment complete


# CELL 7 — Graph 1: Sentiment Distribution (Left vs Right)

In [7]:
print("Generating Graph 1 — Sentiment Distribution (Left vs Right)...")

methods     = ['vader', 'textblob', 'roberta']
method_names = ['VADER', 'TextBlob', 'RoBERTa']
ideologies  = ['Left', 'Right']
colors      = {'Positive': '#4CAF50', 'Negative': '#F44336'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Sentiment Distribution by Political Ideology\n(Left vs Right)',
             fontsize=15, fontweight='bold')

for ax, method, name in zip(axes, methods, method_names):
    col = f'{method}_sentiment'
    data = []
    for ideology in ideologies:
        sub  = df[df['ideology'] == ideology]
        pos  = (sub[col] == 'Positive').sum() / len(sub) * 100
        neg  = (sub[col] == 'Negative').sum() / len(sub) * 100
        data.append([pos, neg])

    x     = np.arange(len(ideologies))
    width = 0.35
    bars1 = ax.bar(x - width/2, [d[0] for d in data], width,
                   label='Positive', color=colors['Positive'], alpha=0.85)
    bars2 = ax.bar(x + width/2, [d[1] for d in data], width,
                   label='Negative', color=colors['Negative'], alpha=0.85)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom',
                fontsize=9, fontweight='bold')
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{bar.get_height():.1f}%', ha='center', va='bottom',
                fontsize=9, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(ideologies, fontsize=12)
    ax.set_ylim(0, 110)
    ax.set_ylabel('Percentage (%)', fontsize=11)
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
g1_path = os.path.join(OUTPUT_PATH, 'graph3_sentiment_distribution.png')
plt.savefig(g1_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Graph 1 saved → {g1_path}")


Generating Graph 1 — Sentiment Distribution (Left vs Right)...
✅ Graph 1 saved → /content/drive/MyDrive/PoliGraphX/outputs/graph3_sentiment_distribution.png


# CELL 8 — Graph 2: Sentiment Over Time

In [8]:
if 'month' not in df.columns or df['month'].isna().all():
    print("⚠️ No date column found — skipping sentiment over time graph")
else:
    print("Generating Graph 2 — Sentiment Over Time...")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Sentiment Over Time by Ideology', fontsize=15, fontweight='bold')

    for ax, method, name in zip(axes, methods, method_names):
        col = f'{method}_sentiment'
        for ideology, color in zip(['Left', 'Right'], ['#2196F3', '#FF5722']):
            sub   = df[df['ideology'] == ideology].copy()
            trend = sub.groupby('month')[col].apply(
                lambda x: (x == 'Positive').sum() / len(x) * 100
            ).reset_index()
            trend.columns = ['month', 'positive_pct']
            trend['month_str'] = trend['month'].astype(str)
            ax.plot(trend['month_str'], trend['positive_pct'],
                    marker='o', label=ideology, color=color, linewidth=2)

        ax.set_title(f'{name}', fontsize=13, fontweight='bold')
        ax.set_ylabel('% Positive Tweets', fontsize=11)
        ax.set_xlabel('Month', fontsize=11)
        ax.legend(fontsize=10)
        ax.grid(alpha=0.3)
        ax.tick_params(axis='x', rotation=45)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()
    g2_path = os.path.join(OUTPUT_PATH, 'graph4_sentiment_over_time.png')
    plt.savefig(g2_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Graph 2 saved → {g2_path}")

Generating Graph 2 — Sentiment Over Time...
✅ Graph 2 saved → /content/drive/MyDrive/PoliGraphX/outputs/graph4_sentiment_over_time.png


# CELL 9 — Graph 3: Word Clouds per Sentiment

In [9]:
print("Generating Graph 3 — Word Clouds per Sentiment...")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Word Clouds by Sentiment & Ideology\n(Using RoBERTa Labels)',
             fontsize=15, fontweight='bold')

combos = [
    ('Left',  'Positive', axes[0][0], '#4CAF50'),
    ('Left',  'Negative', axes[0][1], '#F44336'),
    ('Right', 'Positive', axes[1][0], '#2196F3'),
    ('Right', 'Negative', axes[1][1], '#FF5722'),
]

for ideology, sentiment, ax, color in combos:
    subset = df[(df['ideology'] == ideology) &
                (df['roberta_sentiment'] == sentiment)]
    text   = ' '.join(subset['clean_text'].astype(str).tolist())

    if len(text.strip()) == 0:
        ax.text(0.5, 0.5, 'No Data', ha='center', va='center', fontsize=16)
    else:
        wc = WordCloud(
            width=800, height=400,
            background_color='white',
            colormap='Greens' if sentiment == 'Positive' else 'Reds',
            max_words=100,
            collocations=False
        ).generate(text)
        ax.imshow(wc, interpolation='bilinear')

    ax.set_title(f'{ideology} — {sentiment} ({len(subset):,} tweets)',
                 fontsize=12, fontweight='bold', color=color)
    ax.axis('off')

plt.tight_layout()
g3_path = os.path.join(OUTPUT_PATH, 'graph5_wordclouds.png')
plt.savefig(g3_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Graph 3 saved → {g3_path}")


Generating Graph 3 — Word Clouds per Sentiment...
✅ Graph 3 saved → /content/drive/MyDrive/PoliGraphX/outputs/graph5_wordclouds.png


# CELL 10 — Graph 4: Sentiment Confidence Scores

In [10]:
print("Generating Graph 4 — Sentiment Confidence Scores...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Sentiment Confidence Score Distribution by Ideology',
             fontsize=15, fontweight='bold')

for ax, method, name in zip(axes, methods, method_names):
    conf_col = f'{method}_confidence'
    for ideology, color in zip(['Left', 'Right'], ['#2196F3', '#FF5722']):
        sub = df[df['ideology'] == ideology][conf_col]
        ax.hist(sub, bins=30, alpha=0.6, label=ideology,
                color=color, edgecolor='white')

    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Confidence Score', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
g4_path = os.path.join(OUTPUT_PATH, 'graph6_confidence_scores.png')
plt.savefig(g4_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Graph 4 saved → {g4_path}")

Generating Graph 4 — Sentiment Confidence Scores...
✅ Graph 4 saved → /content/drive/MyDrive/PoliGraphX/outputs/graph6_confidence_scores.png


# CELL 11 — Compare all 3 methods

In [11]:
print("Generating Model Comparison Table...")

rows = []
for method, name in zip(methods, method_names):
    col = f'{method}_sentiment'
    total     = len(df)
    pos_total = (df[col] == 'Positive').sum()
    neg_total = (df[col] == 'Negative').sum()

    left_pos  = (df[df['ideology']=='Left'][col] == 'Positive').sum()
    left_neg  = (df[df['ideology']=='Left'][col] == 'Negative').sum()
    right_pos = (df[df['ideology']=='Right'][col] == 'Positive').sum()
    right_neg = (df[df['ideology']=='Right'][col] == 'Negative').sum()

    rows.append({
        'Method'         : name,
        'Total Positive' : f"{pos_total:,} ({pos_total/total*100:.1f}%)",
        'Total Negative' : f"{neg_total:,} ({neg_total/total*100:.1f}%)",
        'Left Positive'  : f"{left_pos/len(df[df['ideology']=='Left'])*100:.1f}%",
        'Left Negative'  : f"{left_neg/len(df[df['ideology']=='Left'])*100:.1f}%",
        'Right Positive' : f"{right_pos/len(df[df['ideology']=='Right'])*100:.1f}%",
        'Right Negative' : f"{right_neg/len(df[df['ideology']=='Right'])*100:.1f}%",
    })

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

comp_path = os.path.join(OUTPUT_PATH, 'sentiment_comparison.csv')
comparison_df.to_csv(comp_path, index=False)
print(f"\n✅ Saved → {comp_path}")


Generating Model Comparison Table...
  Method Total Positive Total Negative Left Positive Left Negative Right Positive Right Negative
   VADER 26,560 (63.1%) 15,550 (36.9%)         64.2%         35.8%          62.4%          37.6%
TextBlob 32,004 (76.0%) 10,106 (24.0%)         76.7%         23.3%          75.6%          24.4%
 RoBERTa  6,450 (15.3%) 35,660 (84.7%)         19.5%         80.5%          12.9%          87.1%

✅ Saved → /content/drive/MyDrive/PoliGraphX/outputs/sentiment_comparison.csv


# CELL 12 — Save final labeled dataset

In [12]:
final_path = os.path.join(DRIVE_PATH, 'tweets_sentiment_labeled.csv')
df.to_csv(final_path, index=False, encoding='utf-8-sig')
print(f"✅ Final dataset saved → {final_path}")
print(f"   Total tweets : {len(df):,}")
print(f"   Columns      : {list(df.columns)}")

✅ Final dataset saved → /content/drive/MyDrive/PoliGraphX/tweets_sentiment_labeled.csv
   Total tweets : 42,110
   Columns      : ['text', 'rawContent', 'date', 'lang', 'username', 'likeCount', 'retweetCount', 'replyCount', 'quoteCount', 'viewCount', 'hashtags', 'mentionedUsers', 'retweetedTweet', 'epoch', 'id_str', 'location', 'tweet_text', 'date_only', 'year', 'month', 'day', 'hour', 'hashtags_list', 'hashtags_str', 'hashtag_count', 'mentioned_list', 'mention_count', 'clean_text', 'ideology', 'ideology_confidence', 'vader_sentiment', 'vader_confidence', 'textblob_sentiment', 'textblob_confidence', 'roberta_sentiment', 'roberta_confidence']


# CELL 13 — Final Summary

In [13]:
print("=" * 55)
print("NOTEBOOK 3 COMPLETE ✅")
print("=" * 55)
print(f"\nOutputs saved to: {OUTPUT_PATH}")
print(f"  ✅ graph3_sentiment_distribution.png")
print(f"  ✅ graph4_sentiment_over_time.png")
print(f"  ✅ graph5_wordclouds.png")
print(f"  ✅ graph6_confidence_scores.png")
print(f"  ✅ sentiment_comparison.csv")
print(f"\nFinal dataset:")
print(f"  ✅ tweets_sentiment_labeled.csv ({len(df):,} tweets)")
print(f"\nNew columns added:")
print(f"  - vader_sentiment, vader_confidence")
print(f"  - textblob_sentiment, textblob_confidence")
print(f"  - roberta_sentiment, roberta_confidence")
print(f"\n{'='*55}")
print(f"  Ready for Next Step — Network/Graph Analysis")
print(f"{'='*55}")

NOTEBOOK 3 COMPLETE ✅

Outputs saved to: /content/drive/MyDrive/PoliGraphX/outputs
  ✅ graph3_sentiment_distribution.png
  ✅ graph4_sentiment_over_time.png
  ✅ graph5_wordclouds.png
  ✅ graph6_confidence_scores.png
  ✅ sentiment_comparison.csv

Final dataset:
  ✅ tweets_sentiment_labeled.csv (42,110 tweets)

New columns added:
  - vader_sentiment, vader_confidence
  - textblob_sentiment, textblob_confidence
  - roberta_sentiment, roberta_confidence

  Ready for Next Step — Network/Graph Analysis
